# 🚀 CoreOps OpsPilot v3 — Multi-LLM Inference Server

**Architecture: "LLM Thinks, Code Executes"**

| Model | Task | VRAM | Optimization |
|-------|------|------|-------------|
| Phi-3.5-mini | Chat & General | ~2GB | ERP system prompt |
| Qwen3-4B | Intent Classification | ~2.5GB | **LoRA fine-tuned** + structured JSON |
| DeepSeek-R1-Distill-Qwen-7B | Reasoning & Analysis | ~4GB | Data-focused system prompt |

### What's New in v3:
- 🎯 LoRA fine-tuned intent model (Cell 3.5)
- 🧠 ERP-specific system prompts per model
- 🔥 KV-cache warmup for faster first inference
- 📊 Structured JSON output with validation
- ⚡ Optimized `/api/intent` endpoint

> **Run all cells in order.** Cell 3.5 (LoRA training) only needs to run ONCE.

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 1: Install Dependencies (v3 — with LoRA/PEFT)
# ═══════════════════════════════════════════════════════════
!pip install -q flask flask-cors pyngrok
!pip install -q transformers accelerate bitsandbytes
!pip install -q peft trl datasets  # LoRA fine-tuning
!pip install -q google-search-results Pillow
print("✅ All dependencies installed (including LoRA/PEFT)")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 2: Configuration v3 — ERP-Specific System Prompts
# ═══════════════════════════════════════════════════════════
import torch, gc, os, time, json, re

# ┌──────────────────────────────────────────┐
# │   PASTE YOUR TOKENS HERE                 │
# └──────────────────────────────────────────┘
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN"
SERPAPI_KEY      = "YOUR_SERPAPI_KEY"

# ── Model Config with ERP System Prompts ──
MODELS_CONFIG = {
    "chat": {
        "name": "microsoft/Phi-3.5-mini-instruct",
        "task": "General chat & Q&A",
        "max_tokens": 1024,
        "temperature": 0.7,
        "default_system_prompt": (
            "You are OpsPilot, the AI assistant for CoreOps ERP.\n"
            "You help with: assets, inventory, maintenance, purchase orders, budgets, transactions.\n"
            "Always be concise and professional. Format responses with markdown.\n"
            "If asked about capabilities, list: asset management, inventory control, PO approval/rejection,\n"
            "maintenance ticket handling, budget tracking, transaction recording, anomaly detection, and data queries."
        ),
    },
    "intent": {
        "name": "Qwen/Qwen3-4B",
        "task": "Intent classification (LoRA fine-tuned)",
        "max_tokens": 256,
        "temperature": 0.05,
        "lora_adapter": "./coreops-intent-lora",
        "default_system_prompt": (
            "You are an ERP intent classifier for CoreOps.\n"
            "Given a user command, output ONLY valid JSON with this schema:\n"
            '{"intent":"INTENT_NAME","entities":{...},"confidence":0.0-1.0}\n'
            "\n"
            "Valid intents: CREATE_ASSET, CREATE_TRANSACTION, REFILL_INVENTORY, \n"
            "APPROVE_PURCHASE, REJECT_PURCHASE, CLOSE_MAINTENANCE, CREATE_TICKET,\n"
            "UPDATE_ASSET, GET_LOW_STOCK, SET_BUDGET, MATCH_INVOICE, PROCESS_BILL,\n"
            "QUERY_DATA, DETECT_ANOMALY, FORECAST_BUDGET, GENERAL\n"
            "\n"
            "Valid asset categories: LAPTOP, COMPUTER, FURNITURE, VEHICLE, EQUIPMENT,\n"
            "PHONE, PRINTER, SERVER, NETWORK, MACHINERY, OTHER\n"
            "\n"
            "Valid transaction types: INCOME, EXPENSE\n"
            "Valid priorities: LOW, MEDIUM, HIGH, CRITICAL\n"
            "\n"
            "Output ONLY the JSON object, nothing else."
        ),
    },
    "reasoning": {
        "name": "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
        "task": "Deep analysis & data synthesis",
        "max_tokens": 2048,
        "temperature": 0.3,
        "default_system_prompt": (
            "You are OpsPilot Analytics, the AI analyst for CoreOps ERP.\n"
            "You receive live ERP data snapshots and answer questions about:\n"
            "- Budget utilization and forecasts\n"
            "- Transaction anomaly detection\n"
            "- Inventory demand patterns\n"
            "- Asset lifecycle analysis\n"
            "- Maintenance cost trends\n"
            "\n"
            "Rules:\n"
            "1. Always cite specific numbers from the data\n"
            "2. Use INR currency with commas (e.g., 1,25,000)\n"
            "3. Use markdown tables when comparing items\n"
            "4. Flag values >2x average as anomalies\n"
            "5. Never fabricate data — if missing, say so"
        ),
    },
}

# GPU check
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name} | VRAM: {gpu_mem:.1f} GB")
else:
    print("⚠️ No GPU — models will be slow")
    gpu_mem = 0

print(f"📦 Models: {', '.join(MODELS_CONFIG.keys())}")
print(f"🎯 Intent model will use LoRA adapter if available")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 3: Load Models (4-bit quantized + LoRA support)
# ═══════════════════════════════════════════════════════════
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

loaded_models = {}
loaded_tokenizers = {}

def load_model(key):
    """Load a model with 4-bit quantization and optional LoRA adapter."""
    cfg = MODELS_CONFIG[key]
    model_name = cfg["name"]
    print(f"\n{'='*55}")
    print(f"⏳ Loading [{key}]: {model_name}")
    print(f"{'='*55}")
    t0 = time.time()

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.float16,
        )

        # Load LoRA adapter if specified and available
        lora_path = cfg.get("lora_adapter")
        if lora_path and os.path.exists(lora_path):
            from peft import PeftModel
            print(f"  🔧 Loading LoRA adapter from {lora_path}")
            model = PeftModel.from_pretrained(model, lora_path)
            model = model.merge_and_unload()  # Merge for zero-overhead inference
            print(f"  ✅ LoRA merged — model specialized for ERP")

        model.eval()
        loaded_models[key] = model
        loaded_tokenizers[key] = tokenizer

        elapsed = time.time() - t0
        vram_used = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
        print(f"✅ [{key}] loaded in {elapsed:.0f}s | VRAM: {vram_used:.1f}GB")
        return True

    except Exception as e:
        print(f"❌ [{key}] FAILED: {e}")
        return False

# Load smallest → largest
for key in ["chat", "intent", "reasoning"]:
    load_model(key)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Summary
print(f"\n{'='*55}")
print(f"📊 LOADED: {list(loaded_models.keys())}")
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"💾 VRAM: {used:.1f}GB / {total:.1f}GB ({used/total*100:.0f}%)")
print(f"{'='*55}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 3.5: LoRA Fine-Tuning for Intent Model
# ═══════════════════════════════════════════════════════════
# ⚠️ RUN THIS CELL ONCE to fine-tune, then SKIP on future runs.
#    After training, the adapter is saved and auto-loaded in Cell 3.
# ═══════════════════════════════════════════════════════════

TRAIN_DATA_PATH = "/kaggle/input/coreops-training/coreops_intent_train.jsonl"
LORA_OUTPUT_DIR = "./coreops-intent-lora"

# Also check local path for testing
if not os.path.exists(TRAIN_DATA_PATH):
    local_path = "./coreops_intent_train.jsonl"
    if os.path.exists(local_path):
        TRAIN_DATA_PATH = local_path

if os.path.exists(LORA_OUTPUT_DIR):
    print("✅ LoRA adapter already exists — SKIPPING training.")
    print(f"   Delete '{LORA_OUTPUT_DIR}' to retrain.")

elif os.path.exists(TRAIN_DATA_PATH) and "intent" in loaded_models:
    print("🎯 Starting LoRA fine-tuning for ERP intent classification...")
    print(f"   Training data: {TRAIN_DATA_PATH}")
    t0 = time.time()

    from peft import LoraConfig, get_peft_model
    from trl import SFTTrainer, SFTConfig
    from datasets import load_dataset

    # Load training data
    dataset = load_dataset("json", data_files=TRAIN_DATA_PATH, split="train")
    print(f"   📊 Training examples: {len(dataset)}")

    # LoRA config — targets attention layers for maximum impact
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )

    # Apply LoRA to loaded intent model
    intent_model = get_peft_model(loaded_models["intent"], lora_config)
    trainable = sum(p.numel() for p in intent_model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in intent_model.parameters())
    print(f"   🔧 Trainable params: {trainable:,} / {total_params:,} ({100*trainable/total_params:.2f}%)")

    # Training configuration
    training_args = SFTConfig(
        output_dir=LORA_OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        warmup_steps=10,
        fp16=True,
        logging_steps=5,
        save_strategy="epoch",
        max_seq_length=512,
        report_to="none",
    )

    # Train
    trainer = SFTTrainer(
        model=intent_model,
        train_dataset=dataset,
        args=training_args,
        processing_class=loaded_tokenizers["intent"],
    )

    print("   🚀 Training started...")
    trainer.train()
    trainer.save_model(LORA_OUTPUT_DIR)
    loaded_tokenizers["intent"].save_pretrained(LORA_OUTPUT_DIR)

    elapsed = time.time() - t0
    print(f"\n✅ Fine-tuning complete in {elapsed:.0f}s!")
    print(f"   Adapter saved to: {LORA_OUTPUT_DIR}")

    # Reload with merged adapter for inference
    from peft import PeftModel
    loaded_models["intent"] = PeftModel.from_pretrained(
        loaded_models["intent"], LORA_OUTPUT_DIR
    ).merge_and_unload()
    loaded_models["intent"].eval()
    print("   ✅ LoRA adapter merged into intent model")

    # Cleanup
    del trainer, intent_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

else:
    print("⏭️ No training data found — using base model for intent.")
    print(f"   Expected: {TRAIN_DATA_PATH}")
    print("   Upload coreops_intent_train.jsonl to Kaggle Datasets to enable fine-tuning.")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 4: Inference Functions v3 (structured output + warmup)
# ═══════════════════════════════════════════════════════════

def generate_text(model_key, prompt, system_prompt=None, max_tokens=None, temperature=None):
    """Generate text using a loaded model."""
    if model_key not in loaded_models:
        return {"error": f"Model '{model_key}' not loaded", "text": None}

    cfg = MODELS_CONFIG[model_key]
    model = loaded_models[model_key]
    tokenizer = loaded_tokenizers[model_key]
    max_new = max_tokens or cfg["max_tokens"]
    temp = temperature if temperature is not None else cfg["temperature"]

    # Use default system prompt if none provided
    if system_prompt is None:
        system_prompt = cfg.get("default_system_prompt")

    # Build chat messages
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    # Apply chat template
    try:
        text_input = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        if system_prompt:
            text_input = f"System: {system_prompt}\nUser: {prompt}\nAssistant:"
        else:
            text_input = f"User: {prompt}\nAssistant:"

    inputs = tokenizer(text_input, return_tensors="pt", truncation=True, max_length=4096)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[1]

    t0 = time.time()
    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new,
                temperature=max(temp, 0.01),
                do_sample=(temp > 0.01),
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            )

        new_tokens = outputs[0][input_len:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        duration_ms = int((time.time() - t0) * 1000)

        # Strip <think>...</think> blocks from DeepSeek-R1
        think_end = response.find("</think>")
        if think_end != -1:
            response = response[think_end + 8:].strip()
        response = re.sub(r'<think>[\s\S]*?</think>', '', response).strip()

        return {
            "text": response,
            "model": cfg["name"],
            "model_key": model_key,
            "duration_ms": duration_ms,
            "tokens_generated": len(new_tokens),
        }
    except Exception as e:
        return {"error": str(e), "text": None, "model_key": model_key}


def generate_intent(prompt, system_prompt=None):
    """Specialized intent extraction with forced JSON + validation."""
    cfg = MODELS_CONFIG["intent"]
    sys = system_prompt or cfg["default_system_prompt"]

    result = generate_text("intent", prompt, system_prompt=sys,
                          max_tokens=256, temperature=0.05)

    if result.get("error") or not result.get("text"):
        return {**result, "parsed": None}

    text = result["text"]

    # Multi-strategy JSON extraction
    for fn in [
        lambda t: json.loads(t),
        lambda t: json.loads(re.search(r'```json\s*([\s\S]*?)```', t).group(1)),
        lambda t: json.loads(re.search(r'(\{[\s\S]*\})', t).group(1)),
    ]:
        try:
            parsed = fn(text)
            # Validate required fields
            if "intent" not in parsed:
                parsed["intent"] = "GENERAL"
            if "entities" not in parsed:
                parsed["entities"] = {}
            if "confidence" not in parsed:
                parsed["confidence"] = 0.5
            return {**result, "parsed": parsed}
        except Exception:
            continue

    # All strategies failed — return GENERAL
    return {**result, "parsed": {"intent": "GENERAL", "entities": {}, "confidence": 0.0}}


def generate_json(model_key, prompt, system_prompt=None):
    """Generate text and parse JSON from it."""
    result = generate_text(model_key, prompt, system_prompt=system_prompt)
    if result.get("error") or not result.get("text"):
        return {**result, "parsed": None}

    text = result["text"]
    for fn in [
        lambda t: json.loads(t),
        lambda t: json.loads(re.search(r'```json\s*([\s\S]*?)```', t).group(1)),
        lambda t: json.loads(re.search(r'(\{[\s\S]*\})', t).group(1)),
    ]:
        try:
            return {**result, "parsed": fn(text)}
        except Exception:
            continue
    return {**result, "parsed": None}


def search_web(query):
    """Search internet via SerpAPI."""
    if not SERPAPI_KEY or SERPAPI_KEY == "YOUR_SERPAPI_KEY":
        return {"error": "SerpAPI not configured", "results": []}
    try:
        from serpapi import GoogleSearch
        results = GoogleSearch({"q": query, "api_key": SERPAPI_KEY, "num": 5}).get_dict()
        organic = results.get("organic_results", [])
        return {
            "results": [{"title": r.get("title"), "snippet": r.get("snippet"), "link": r.get("link")} for r in organic[:5]],
            "query": query,
        }
    except Exception as e:
        return {"error": str(e), "results": []}


# ── KV-Cache Warmup ──
def warmup_models():
    """Pre-fill KV caches so first real request is fast."""
    print("🔥 Warming up model caches...")
    for key in loaded_models:
        generate_text(key, "Hello", max_tokens=5)
        print(f"   ✅ {key} warmed")
    print("✅ All models warmed up — first requests will be fast!")

warmup_models()

# Quick accuracy test
if "intent" in loaded_models:
    test = generate_intent("Create a MacBook Air M3 asset costing 1200")
    parsed = test.get("parsed", {})
    intent = parsed.get("intent", "UNKNOWN")
    print(f"\n🧪 Intent test: '{intent}' (expected: CREATE_ASSET)")
    print(f"   Entities: {parsed.get('entities', {})}")
    print(f"   Time: {test.get('duration_ms', '?')}ms")
    if intent == "CREATE_ASSET":
        print("   ✅ PASS")
    else:
        print("   ⚠️ Intent mismatch — base model needs fine-tuning")

print("\n✅ Inference functions ready")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 5: Flask API Server v3 (Structured Intent + ERP Context)
# ═══════════════════════════════════════════════════════════
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

@app.errorhandler(Exception)
def handle_error(e):
    return jsonify({"success": False, "error": str(e)}), 500

# ─── Health ──────────────────────────────────
@app.route("/api/health", methods=["GET"])
def health():
    vram_used = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
    return jsonify({
        "status": "ok",
        "version": "v3-lora",
        "models_loaded": list(loaded_models.keys()),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "vram": {"used_gb": round(vram_used, 1), "total_gb": round(vram_total, 1)},
        "lora_fine_tuned": os.path.exists("./coreops-intent-lora"),
        "search_enabled": SERPAPI_KEY != "YOUR_SERPAPI_KEY",
    })

# ─── Reasoning ──────────────────────────────
@app.route("/api/reasoning", methods=["POST"])
def api_reasoning():
    data = request.get_json(force=True)
    result = generate_text(
        "reasoning",
        data.get("prompt", ""),
        system_prompt=data.get("system_prompt"),
        max_tokens=data.get("max_tokens"),
        temperature=data.get("temperature"),
    )
    return jsonify({"success": not result.get("error"), "data": result})

# ─── Intent (v3 — Structured JSON extraction) ──────────
@app.route("/api/intent", methods=["POST"])
def api_intent():
    data = request.get_json(force=True)
    result = generate_intent(
        data.get("prompt", ""),
        system_prompt=data.get("system_prompt"),
    )
    return jsonify({"success": result.get("parsed") is not None, "data": result})

# ─── Chat ────────────────────────────────────
@app.route("/api/chat", methods=["POST"])
def api_chat():
    data = request.get_json(force=True)
    result = generate_text(
        "chat",
        data.get("prompt", ""),
        system_prompt=data.get("system_prompt"),
        max_tokens=data.get("max_tokens"),
        temperature=data.get("temperature"),
    )
    return jsonify({"success": not result.get("error"), "data": result})

# ─── Vision (placeholder) ─────────────────────
@app.route("/api/vision", methods=["POST"])
def api_vision():
    return jsonify({"success": False, "data": {"error": "Vision model not loaded", "text": None}})

# ─── Search ──────────────────────────────────
@app.route("/api/search", methods=["POST"])
def api_search():
    data = request.get_json(force=True)
    query = data.get("query", "")
    result = search_web(query)
    if data.get("summarize") and result.get("results"):
        snippets = "\n".join([f"- {r['title']}: {r['snippet']}" for r in result["results"]])
        summary = generate_text("chat", f"Summarize about '{query}':\n{snippets}")
        result["summary"] = summary.get("text")
    return jsonify({"success": not result.get("error"), "data": result})

# ─── Orchestrate v3 (Smart routing with ERP context) ─────
@app.route("/api/orchestrate", methods=["POST"])
def api_orchestrate():
    data = request.get_json(force=True)
    prompt = data.get("prompt", "")
    erp_context = data.get("erp_context", "")  # ERP snapshot from backend

    # Step 1: Classify intent (fast, ~2-5s with fine-tuned model)
    intent_result = generate_intent(prompt)
    parsed = intent_result.get("parsed") or {"intent": "GENERAL"}
    intent_name = parsed.get("intent", "GENERAL")

    # Step 2: Route based on intent
    action_intents = [
        "CREATE_ASSET", "UPDATE_ASSET", "CREATE_TRANSACTION", "REFILL_INVENTORY",
        "APPROVE_PURCHASE", "REJECT_PURCHASE", "CLOSE_MAINTENANCE", "CREATE_TICKET",
        "GET_LOW_STOCK", "GET_ASSET_STATS", "SET_BUDGET", "MATCH_INVOICE", "PROCESS_BILL",
    ]

    if intent_name in ["QUERY_DATA", "DETECT_ANOMALY", "FORECAST_BUDGET"]:
        # Use reasoning model WITH ERP context
        analysis_prompt = f"ERP Data Snapshot:\n{erp_context}\n\nUser Question: {prompt}\n\nAnalyze the data and provide a detailed answer."
        main_result = generate_text("reasoning", analysis_prompt)
    elif intent_name in action_intents:
        # Action intents — backend handles execution, just return parsed intent
        main_result = {"text": f"Action '{intent_name}' dispatched to backend executor.", "duration_ms": 0, "model_key": "none"}
    elif intent_name == "GENERAL":
        main_result = generate_text("chat", prompt)
    else:
        main_result = generate_text("chat", prompt)

    return jsonify({
        "success": True,
        "data": {
            "intent": intent_name,
            "confidence": parsed.get("confidence", 0),
            "entities": parsed.get("entities", {}),
            "response": main_result.get("text"),
            "model_used": main_result.get("model_key", "chat"),
            "duration_ms": (intent_result.get("duration_ms", 0) or 0) + (main_result.get("duration_ms", 0) or 0),
            "models_chain": ["intent", main_result.get("model_key", "chat")],
            "lora_fine_tuned": os.path.exists("./coreops-intent-lora"),
        }
    })

print("✅ Flask API v3 ready (with structured intent + ERP orchestration)")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 6: START SERVER — Copy the URL to your .env file!
# ═══════════════════════════════════════════════════════════
from pyngrok import ngrok
import threading

if NGROK_AUTH_TOKEN == "YOUR_NGROK_AUTH_TOKEN":
    print("❌ Set your NGROK_AUTH_TOKEN in Cell 2 first!")
    print("   Get it from: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    port = 5001

    # Kill existing tunnels
    for tunnel in ngrok.get_tunnels():
        ngrok.disconnect(tunnel.public_url)

    public_url = ngrok.connect(port).public_url

    print("\n" + "═" * 60)
    print("  🚀  OpsPilot AI Server v3 (LoRA-Enhanced) is LIVE!")
    print("═" * 60)
    print(f"")
    print(f"  📡 URL: {public_url}")
    print(f"")
    print(f"  Add to your backend .env file:")
    print(f"  KAGGLE_INFERENCE_URL={public_url}")
    print(f"")
    print(f"  Models: {list(loaded_models.keys())}")
    print(f"  LoRA: {'✅ Fine-tuned' if os.path.exists('./coreops-intent-lora') else '❌ Base model'}")
    print(f"")
    print(f"  Endpoints:")
    print(f"    POST /api/reasoning  — Deep analysis")
    print(f"    POST /api/intent     — Structured intent extraction")
    print(f"    POST /api/chat       — General conversation")
    print(f"    POST /api/orchestrate — Full pipeline")
    print(f"    POST /api/search     — Web search")
    print(f"    GET  /api/health     — Status check")
    print("═" * 60)

    # Run Flask in background thread
    threading.Thread(
        target=lambda: app.run(host="0.0.0.0", port=port, debug=False, use_reloader=False),
        daemon=True,
    ).start()

    print(f"\n✅ Server running on port {port}")
    print("   Keep this notebook running! Server stops when you close it.")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 7: Test Endpoints (run AFTER Cell 6)
# ═══════════════════════════════════════════════════════════
import requests

BASE = "http://localhost:5001"

# Health
print("\n─── Health ───")
r = requests.get(f"{BASE}/api/health")
h = r.json()
print(f"  Status: {h.get('status')} | Version: {h.get('version')}")
print(f"  Models: {h.get('models_loaded')}")
print(f"  LoRA: {h.get('lora_fine_tuned')}")

# Intent tests
print("\n─── Intent Classification Tests ───")
test_cases = [
    ("Create a Dell laptop asset costing 85000", "CREATE_ASSET"),
    ("Approve purchase order PO-2026-0001", "APPROVE_PURCHASE"),
    ("What is our budget this month?", "QUERY_DATA"),
    ("Refill printer paper to 100 units", "REFILL_INVENTORY"),
    ("Close maintenance ticket MT-0042", "CLOSE_MAINTENANCE"),
    ("Record an expense of 15000 for office supplies", "CREATE_TRANSACTION"),
    ("Hello, what can you do?", "GENERAL"),
]

passed = 0
for prompt, expected in test_cases:
    r = requests.post(f"{BASE}/api/intent", json={"prompt": prompt})
    d = r.json()
    actual = d.get("data", {}).get("parsed", {}).get("intent", "UNKNOWN")
    ms = d.get("data", {}).get("duration_ms", "?")
    status = "✅" if actual == expected else "❌"
    if actual == expected:
        passed += 1
    print(f"  {status} [{ms}ms] '{prompt[:40]}...' → {actual} (expected: {expected})")

print(f"\n  Score: {passed}/{len(test_cases)} ({100*passed//len(test_cases)}%)")

# Chat test
print("\n─── Chat ───")
r = requests.post(f"{BASE}/api/chat", json={"prompt": "What can you help me with?"})
d = r.json()
print(f"  Response: {d.get('data', {}).get('text', '?')[:150]}...")
print(f"  Time: {d.get('data', {}).get('duration_ms', '?')}ms")

print("\n✅ All tests complete!")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 8: Keep Alive (prevents Kaggle timeout)
# ═══════════════════════════════════════════════════════════
import time

print("⏰ Keep-alive running. Server will stay active.")
print("   Press Stop to shut down.\n")

try:
    while True:
        time.sleep(60)
        try:
            r = requests.get(f"http://localhost:5001/api/health", timeout=5)
            status = "✅" if r.status_code == 200 else "⚠️"
        except:
            status = "❌"
        print(f"{status} Alive — {time.strftime('%H:%M:%S')}", end="\r")
except KeyboardInterrupt:
    print("\n🛑 Server stopped.")